# ?? SHE Full-Song Film Master Render (7:25 | 10,680 Frames | 1080p)
### Resumable 10-Chunk Pipeline, Pre-Render Quality Gates, & Google Drive Checkpointing

**Canonical project:** [`sayan-bhattacharya/aianimationautomate`](https://github.com/sayan-bhattacharya/aianimationautomate). SHE is maintained here as a standalone music-film project. It is separate from the NOIR film repository and does not depend on NOIR code or assets.

This render incorporates the reviewed scene revisions: mushroom scale and grounded roots, a slower tree sway, a dinosaur aerial on the opening CRT feed, restrained analog noise on the far-right CRT feed, quieter water reflections, the 7:05 Buddha/CRT placement, and the 7:21 `.STORE` end-card timing.

Run notebook cells in order in a fresh runtime. Chunk files are isolated by a fingerprint of render inputs tracked in the cloned repository and reused only when their manifest, checksum, exact frame count, resolution, codec, and fps validate. The final contact sheet and decode/stream checks run before download. The full-length 1080p master has not been rendered by this notebook yet.


## Step 1: Install Dependencies & Setup Environment
Installs `ffmpeg`, `opencv-python`, `librosa`, `Pillow`, and verifies hardware acceleration.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q opencv-python pillow numpy librosa

import os
import sys
import multiprocessing as mp
import subprocess

print(f"✓ CPU cores available: {mp.cpu_count()}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Running on high-core CPU/TPU instance"


## Step 1b: Mount Google Drive (Recommended)
Mount your Google Drive to store completed chunks and the final 1080p master video directly in your Drive. If Colab disconnects, your progress is permanently preserved.

In [ ]:
# --- MOUNT GOOGLE DRIVE ---
from google.colab import drive

drive.mount('/content/drive')
DRIVE_RENDER_ROOT = '/content/drive/MyDrive/SHE_Film_Renders'
os.makedirs(DRIVE_RENDER_ROOT, exist_ok=True)
print('Google Drive mounted; revision-specific render folders will be created after the source commit is identified.')


## Step 2: Pull SHE Code and Assets from the Canonical Repository
The notebook, renderer, prepared sprites, master audio, and audio features all come from `aianimationautomate` `main`. This keeps the Colab render self-contained and independent of the separate NOIR project.


In [ ]:
import os
import sys
import shutil
import subprocess
import importlib
import hashlib

REPO_DIR = '/content/aianimationautomate'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
print('Cloning canonical SHE project: aianimationautomate/main...')
subprocess.run([
    'git','clone','--depth','1','--branch','main','--single-branch',
    'https://github.com/sayan-bhattacharya/aianimationautomate.git',
    REPO_DIR
],check=True)

RENDER_REVISION = subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()
input_objects = subprocess.check_output([
    'git','-C',REPO_DIR,'rev-parse','HEAD:she_film','HEAD:assets',
    'HEAD:She.wav','HEAD:audio_features_24fps.npy',
    'HEAD:SHE_Full_Song_Master_Render_Colab.ipynb'
],text=True).split()
RENDER_FINGERPRINT = hashlib.sha256('\n'.join(input_objects).encode('utf-8')).hexdigest()
RENDER_KEY = f'{RENDER_FINGERPRINT[:12]}_v2'
OUTPUT_DIR = f'/content/she_render_1080p_{RENDER_KEY}'
DRIVE_RENDER_DIR = os.path.join(DRIVE_RENDER_ROOT, f'SHE_Film_Render_{RENDER_KEY}') if os.path.exists('/content/drive/MyDrive') else None
if DRIVE_RENDER_DIR:
    os.makedirs(DRIVE_RENDER_DIR, exist_ok=True)
    print('Revision-specific Drive checkpoint folder:', DRIVE_RENDER_DIR)
print('Pinned source commit:', RENDER_REVISION)
print('Render-input fingerprint:', RENDER_FINGERPRINT)

required = [
    os.path.join(REPO_DIR, 'she_film'),
    os.path.join(REPO_DIR, 'assets'),
    os.path.join(REPO_DIR, 'She.wav'),
    os.path.join(REPO_DIR, 'audio_features_24fps.npy'),
]
missing = [path for path in required if not os.path.exists(path)]
if missing:
    raise FileNotFoundError('Canonical repository is missing required render inputs: ' + ', '.join(missing))

for source, target in [
    (os.path.join(REPO_DIR, 'she_film'), '/content/she_film'),
    (os.path.join(REPO_DIR, 'assets'), '/content/assets'),
]:
    if os.path.exists(target):
        shutil.rmtree(target)
    shutil.copytree(source, target)
shutil.copy2(os.path.join(REPO_DIR, 'She.wav'), '/content/She.wav')
shutil.copy2(os.path.join(REPO_DIR, 'audio_features_24fps.npy'), '/content/audio_features_24fps.npy')

while '/content' in sys.path:
    sys.path.remove('/content')
sys.path.insert(0, '/content')
for name in list(sys.modules):
    if name == 'she_film' or name.startswith('she_film.'):
        del sys.modules[name]
importlib.invalidate_caches()

from she_film import clock, generate_screens_cache, plates
if abs(clock.DURATION - 445.0) > 1e-6 or clock.TOTAL_FRAMES != 10680:
    raise RuntimeError(f'Loaded repository code does not match this 7:25 notebook: DURATION={clock.DURATION}, TOTAL_FRAMES={clock.TOTAL_FRAMES}')

# Build the directed dinosaur aerial and restrained far-right analog-noise feed.
active_package = os.path.dirname(os.path.abspath(generate_screens_cache.__file__))
plates_package = os.path.dirname(os.path.abspath(plates.__file__))
if os.path.realpath(active_package) != os.path.realpath(plates_package):
    raise RuntimeError(f'Package path mismatch: generator={active_package}, plates={plates_package}')
screens_path = generate_screens_cache.build_screens_cache(plates.CACHE)
import numpy as np
with np.load(screens_path) as screens:
    for key in ('scr_left','scr_mid_left','scr_mid_right','scr_right'):
        if key not in screens or screens[key].ndim != 4 or len(screens[key]) < 240 or screens[key].shape[1:] != (320,240,3):
            raise RuntimeError(f'Invalid or missing screen feed: {key}')
        movement = float(np.mean(np.abs(screens[key][0].astype(np.float32)-screens[key][len(screens[key])//2].astype(np.float32))))
        if movement <= 2.0:
            raise RuntimeError(f'Static CRT feed: {key}; midpoint difference={movement:.2f}')
print('Canonical repository:', REPO_DIR)
print('SHE renderer:', active_package)
print('Prepared assets:', os.path.join(REPO_DIR, 'assets'))
print('Master audio: /content/She.wav')
print('Audio features: /content/audio_features_24fps.npy')
print('Four moving CRT feeds validated:', screens_path)


## Art Direction and Scene Coupling

Keep the camera and horizon quiet. Let only a few low-contrast water crests move, and treat light as a gradual shift in perception. The dinosaurs should read as distant, monumental presences. Plant forms emerge slowly with the music, hold long enough to feel calm, then recede; avoid cutting on every beat.

The ending preserves the requested story order: at approximately **7:05**, the Buddha tableau shows the dinosaur aerial on the first CRT screen and subdued analog noise at the far right; at approximately **7:21**, the closing credit includes `.STORE`.

This is a visual direction note for SHE only. The NOIR film has its own repository and materials.


## Step 3: Run Pre-Render Quality Gatekeeper Review
**MANDATORY**: Before spending compute on the full render, this automated reviewer audits all 7 film segments, testing lifecycle timing, pterodactyl continuity, celestial lighting/reflections, CRT placement, and four moving CRT feeds. Code checks catch timing and asset defects; they do not replace visual review. Rendering is blocked if any hard check fails.


In [ ]:
from she_film import pre_render_code_review

passed = pre_render_code_review.run_pre_render_code_review()
from IPython.display import display, Image as NotebookImage
debug_image = '/content/preview_720p/tv_screen_alignment_debug.png'
if os.path.exists(debug_image):
    display(NotebookImage(filename=debug_image))
if not passed:
    raise RuntimeError('❌ PRE-RENDER GATEKEEPER FAILED: Code review found defects. Halting execution.')
print('\n>>> ALL PRE-RENDER CHECKS PASSED: Cleared for 1080p Master Render! <<<')


## Step 4: Full-Song 1080p Parallel Render Pipeline
Renders the complete 10,680 frames in 10 sequential, verified chunks to keep memory use predictable. The chunk selector rebuilds only changed intervals and reuses verified unaffected chunks; each completed render is backed up to Google Drive.

In [ ]:
import cv2
import time
import json
import hashlib
from fractions import Fraction
from she_film import frame, plates
from she_film.clock import DURATION, FPS, TOTAL_FRAMES

if 'RENDER_REVISION' not in globals() or 'RENDER_KEY' not in globals():
    raise RuntimeError('Run the canonical repository setup cell before rendering.')
OUTPUT_DIR = f'/content/she_render_1080p_{RENDER_KEY}'
CHUNKS_DIR = os.path.join(OUTPUT_DIR, 'chunks')
os.makedirs(CHUNKS_DIR, exist_ok=True)
DRIVE_BACKUP = os.path.join(DRIVE_RENDER_DIR, 'chunks') if DRIVE_RENDER_DIR else None
if DRIVE_BACKUP:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print('Revision-specific Drive checkpoint active:', DRIVE_BACKUP)

NUM_CHUNKS = 10
FORCE_FRESH_RENDER = False  # Set True only to rebuild this exact revision's verified chunks.
RENDER_CONFIG = {'render_fingerprint': RENDER_FINGERPRINT, 'render_key': RENDER_KEY,
                 'width': 1920, 'height': 1080, 'fps': FPS,
                 'duration': DURATION, 'total_frames': TOTAL_FRAMES,
                 'codec': 'libx264', 'pixel_format': 'yuv420p',
                 'preset': 'ultrafast', 'crf': 18,
                 'chunk_count': NUM_CHUNKS}

chunk_ranges = []
chunk_sz = TOTAL_FRAMES // NUM_CHUNKS
for i in range(NUM_CHUNKS):
    f_start = i * chunk_sz
    f_end = (i + 1) * chunk_sz if i < NUM_CHUNKS - 1 else TOTAL_FRAMES
    chunk_ranges.append((i + 1, f_start, f_end))
print(f'Plan: {NUM_CHUNKS} sequential chunks | {TOTAL_FRAMES} frames | 1920x1080 | {FPS} fps | source {RENDER_KEY}')
for c_id, f0, f1 in chunk_ranges:
    print(f'  Chunk {c_id:02d}: {f0:05d}-{f1:05d} ({f1-f0} frames; {f0/FPS:.1f}-{f1/FPS:.1f}s)')

print('Loading shared plates...')
p = plates.load()
print('Plates loaded.')

def _sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _manifest_path(path):
    return path + '.manifest.json'

def _chunk_record(path, c_id, f_start, f_end):
    with open(_manifest_path(path), encoding='utf-8') as handle:
        record = json.load(handle)
    expected = {'config': RENDER_CONFIG, 'chunk_id': c_id,
                'frame_start': f_start, 'frame_end': f_end,
                'expected_frames': f_end - f_start}
    for key, value in expected.items():
        if record.get(key) != value:
            raise ValueError(f'{key} mismatch in {os.path.basename(path)}')
    if record.get('sha256') != _sha256(path):
        raise ValueError('chunk checksum mismatch')
    probe = subprocess.run([
        'ffprobe','-v','error','-select_streams','v:0',
        '-show_entries','stream=codec_name,width,height,avg_frame_rate,nb_frames,duration',
        '-of','json',path
    ],check=True,capture_output=True,text=True)
    streams = json.loads(probe.stdout).get('streams', [])
    if not streams:
        raise ValueError('no video stream')
    stream = streams[0]
    fps = float(Fraction(stream.get('avg_frame_rate', '0/1')))
    frames = int(stream.get('nb_frames') or -1)
    if (stream.get('codec_name') != 'h264' or int(stream.get('width', 0)) != 1920 or
        int(stream.get('height', 0)) != 1080 or abs(fps - FPS) > 1e-6 or
        frames != f_end - f_start):
        raise ValueError(f'invalid stream metadata: {stream}')
    return record

def validate_chunk(path, c_id, f_start, f_end):
    if not path or not os.path.isfile(path) or not os.path.isfile(_manifest_path(path)):
        return False, 'chunk or manifest missing'
    try:
        _chunk_record(path, c_id, f_start, f_end)
        return True, 'revision, checksum, frame count, size, codec and fps verified'
    except Exception as exc:
        return False, str(exc)

def _save_manifest(path, c_id, f_start, f_end):
    record = {'config': RENDER_CONFIG, 'chunk_id': c_id,
              'frame_start': f_start, 'frame_end': f_end,
              'expected_frames': f_end - f_start, 'sha256': _sha256(path)}
    temp = _manifest_path(path) + '.tmp'
    with open(temp, 'w', encoding='utf-8') as handle:
        json.dump(record, handle, sort_keys=True, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temp, _manifest_path(path))

def _reuse_candidate(path, local_path, c_id, f_start, f_end):
    ok, reason = validate_chunk(path, c_id, f_start, f_end)
    if not ok:
        if os.path.exists(path):
            print(f'Reject cached chunk {os.path.basename(path)}: {reason}')
        return False
    if os.path.abspath(path) != os.path.abspath(local_path):
        shutil.copy2(path, local_path)
        shutil.copy2(_manifest_path(path), _manifest_path(local_path))
    print(f'Reuse verified chunk {c_id:02d}: {reason}')
    return True

def render_chunk(c_info):
    c_id, f_start, f_end = c_info
    n_chunk_frames = f_end - f_start
    chunk_path = os.path.join(CHUNKS_DIR, f'chunk_{c_id:02d}.mp4')
    drive_chunk = os.path.join(DRIVE_BACKUP, f'chunk_{c_id:02d}.mp4') if DRIVE_BACKUP else None
    if not FORCE_FRESH_RENDER:
        for candidate in (chunk_path, drive_chunk):
            if candidate and _reuse_candidate(candidate, chunk_path, c_id, f_start, f_end):
                return chunk_path

    partial_path = chunk_path.replace('.mp4', '.partial.mp4')
    log_path = partial_path + '.ffmpeg.log'
    video_cmd = ['ffmpeg','-y','-loglevel','error','-f','rawvideo','-vcodec','rawvideo',
                 '-s','1920x1080','-pix_fmt','rgb24','-r',str(FPS),'-i','-',
                 '-an','-c:v','libx264','-preset','ultrafast','-crf','18',
                 '-pix_fmt','yuv420p','-movflags','+faststart',partial_path]
    print(f'\nRender chunk {c_id:02d}: frames {f_start}-{f_end} ({n_chunk_frames} frames)')
    with open(log_path, 'w', encoding='utf-8') as log_handle:
        writer = subprocess.Popen(video_cmd, stdin=subprocess.PIPE, stderr=log_handle)
        t0 = time.time()
        try:
            for idx in range(n_chunk_frames):
                frame_index = f_start + idx
                rendered = frame.render(p, frame_index / FPS)
                writer.stdin.write(rendered.tobytes())
                if (idx + 1) % 200 == 0 or idx == n_chunk_frames - 1:
                    rate = (idx + 1) / max(0.01, time.time() - t0)
                    print(f'  Chunk {c_id:02d}: {idx+1}/{n_chunk_frames} ({(idx+1)/n_chunk_frames*100:.1f}%) | {rate:.1f} fps')
            writer.stdin.close()
            encode_status = writer.wait()
        except BaseException:
            if writer.poll() is None:
                writer.kill()
                writer.wait()
            if writer.stdin and not writer.stdin.closed:
                writer.stdin.close()
            raise
    if encode_status != 0:
        raise RuntimeError(f'ffmpeg failed for chunk {c_id:02d}; inspect {log_path}')
    os.replace(partial_path, chunk_path)
    _save_manifest(chunk_path, c_id, f_start, f_end)
    ok, reason = validate_chunk(chunk_path, c_id, f_start, f_end)
    if not ok:
        for path in (chunk_path, _manifest_path(chunk_path)):
            if os.path.exists(path): os.remove(path)
        raise RuntimeError(f'Fresh chunk {c_id:02d} failed validation: {reason}')
    if drive_chunk:
        shutil.copy2(chunk_path, drive_chunk)
        shutil.copy2(_manifest_path(chunk_path), _manifest_path(drive_chunk))
        ok, reason = validate_chunk(drive_chunk, c_id, f_start, f_end)
        if not ok:
            raise RuntimeError(f'Drive backup validation failed for chunk {c_id:02d}: {reason}')
        print(f'  Verified Drive backup: {drive_chunk}')
    return chunk_path

rendered_chunks = [render_chunk(info) for info in chunk_ranges]
print(f'All {len(rendered_chunks)} revision-matched chunks are validated.')

## Step 5: Master Audio Muxing, Concatenation & SHE Crystal Nocturne Finish
Stitches the 10 verified H.264 chunks with video stream copy, muxes `She.wav` with the exact 7:25 fade, and writes the 1080p master directly into your Google Drive without a second full-length video encode.

The final cell also builds a 50-second **16:9 landscape** Instagram teaser from the rendered master. It preserves the wide composition by request; vertical Reels playback will show a wide video inside the tall phone screen rather than filling it.


In [ ]:
import os
import glob
import shutil
import subprocess
import cv2
import json

if 'chunk_ranges' not in globals() or 'validate_chunk' not in globals():
    raise RuntimeError('Run the render setup and chunk-planning cell before stitching.')
OUTPUT_DIR = f'/content/she_render_1080p_{RENDER_KEY}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
local_chunks = os.path.join(OUTPUT_DIR, 'chunks')
drive_chunks = os.path.join(DRIVE_RENDER_DIR, 'chunks') if DRIVE_RENDER_DIR else None
rendered_chunks = []
for c_id, f_start, f_end in chunk_ranges:
    name = f'chunk_{c_id:02d}.mp4'
    local_path = os.path.join(local_chunks, name)
    drive_path = os.path.join(drive_chunks, name) if drive_chunks else None
    chosen = None
    for candidate in (local_path, drive_path):
        if candidate:
            ok, reason = validate_chunk(candidate, c_id, f_start, f_end)
            if ok:
                chosen = candidate
                break
            if os.path.exists(candidate):
                print(f'Reject {candidate}: {reason}')
    if chosen is None:
        raise FileNotFoundError(f'No valid chunk {c_id:02d} for source revision {RENDER_KEY}. Rerun the chunk render cell.')
    if os.path.abspath(chosen) != os.path.abspath(local_path):
        os.makedirs(local_chunks, exist_ok=True)
        shutil.copy2(chosen, local_path)
        shutil.copy2(chosen + '.manifest.json', local_path + '.manifest.json')
    rendered_chunks.append(local_path)
if len(rendered_chunks) != len(chunk_ranges):
    raise RuntimeError(f'Expected {len(chunk_ranges)} chunks, found {len(rendered_chunks)}')
print('All chunks are ordered and validated for', RENDER_KEY)

concat_list_file = os.path.join(OUTPUT_DIR, 'concat_chunks.txt')
with open(concat_list_file, 'w', encoding='utf-8') as handle:
    for path in rendered_chunks:
        handle.write(f"file '{path}'\n")
FINAL_MASTER_MP4 = os.path.join(OUTPUT_DIR, 'she_full_song_1080p_crystal_nocturne_master.mp4')
AUDIO_WAV = '/content/She.wav'
if not os.path.isfile(AUDIO_WAV):
    raise FileNotFoundError('Master audio /content/She.wav is missing; rerun canonical setup.')
cmd_concat = [
    'ffmpeg','-y','-f','concat','-safe','0','-i',concat_list_file,
    '-i',AUDIO_WAV,'-c:v','copy','-c:a','aac','-b:a','320k',
    '-t',f'{DURATION:.3f}',
    '-af',f'afade=t=in:ss=0:d=1.2,afade=t=out:st={DURATION-12:.3f}:d=12',
    '-shortest',FINAL_MASTER_MP4
]
print('Concatenating validated H.264 chunks and muxing the soundtrack...')
subprocess.run(cmd_concat, check=True)
print('Master assembled:', FINAL_MASTER_MP4)
print(f'File size: {os.path.getsize(FINAL_MASTER_MP4)/1024/1024:.1f} MiB')
if DRIVE_RENDER_DIR:
    drive_master = os.path.join(DRIVE_RENDER_DIR, os.path.basename(FINAL_MASTER_MP4))
    shutil.copy2(FINAL_MASTER_MP4, drive_master)
    print('Master backed up to Drive:', drive_master)

# Landscape teaser: seven authored excerpts, 50 seconds after six 0.5 s crossfades.
TEASER_MP4 = os.path.join(OUTPUT_DIR, 'SHE_Instagram_Landscape_Teaser_1080p_30fps.mp4')
TEASER_PARTS = [(18.0,7.0),(112.0,8.0),(176.0,8.0),(248.0,8.0),(315.0,8.0),(408.0,8.0),(439.0,6.0)]
TEASER_FADE = 0.5
n_parts = len(TEASER_PARTS)
filters = ['[0:v]split='+str(n_parts)+''.join(f'[vsrc{i}]' for i in range(n_parts)),
           '[0:a]asplit='+str(n_parts)+''.join(f'[asrc{i}]' for i in range(n_parts))]
for i,(start,duration) in enumerate(TEASER_PARTS):
    filters.append(f'[vsrc{i}]trim=start={start}:duration={duration},setpts=PTS-STARTPTS,fps=30,format=yuv420p[v{i}]')
    filters.append(f'[asrc{i}]atrim=start={start}:duration={duration},asetpts=PTS-STARTPTS[a{i}]')
v_prev,a_prev='v0','a0'
offset=TEASER_PARTS[0][1]-TEASER_FADE
for i in range(1,n_parts):
    v_next,a_next=f'vx{i}',f'ax{i}'
    filters.append(f'[{v_prev}][v{i}]xfade=transition=fade:duration={TEASER_FADE}:offset={offset:.3f}[{v_next}]')
    filters.append(f'[{a_prev}][a{i}]acrossfade=d={TEASER_FADE}:c1=tri:c2=tri[{a_next}]')
    v_prev,a_prev=v_next,a_next
    offset += TEASER_PARTS[i][1]-TEASER_FADE
cmd_teaser=['ffmpeg','-y','-loglevel','error','-i',FINAL_MASTER_MP4,
            '-filter_complex',';'.join(filters),'-map',f'[{v_prev}]','-map',f'[{a_prev}]',
            '-c:v','libx264','-preset','ultrafast','-crf','18','-pix_fmt','yuv420p',
            '-c:a','aac','-b:a','192k','-movflags','+faststart',TEASER_MP4]
print('Rendering 50-second landscape teaser...')
subprocess.run(cmd_teaser,check=True)
print('Landscape teaser:',TEASER_MP4,'| 1920x1080 | 30 fps | 16:9')
if DRIVE_RENDER_DIR:
    shutil.copy2(TEASER_MP4, os.path.join(DRIVE_RENDER_DIR, os.path.basename(TEASER_MP4)))
    print('Landscape teaser backed up to Drive.')

## Step 6: Verify Final Master Integrity & Direct Download
Checks frame count (must be exactly 10,680), stream duration, and offers direct browser download.

In [ ]:
# Keep the quick integrity readout here. The later review cell is the release gate
# and only starts a download after all required technical checks pass.
import os
import cv2

cap = cv2.VideoCapture(FINAL_MASTER_MP4)
actual_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
actual_fps = float(cap.get(cv2.CAP_PROP_FPS))
cap.release()
actual_duration = actual_frames / actual_fps if actual_fps > 0 else 0.0
print('Quick master readout:', actual_frames, 'frames |', actual_fps, 'fps |', f'{actual_duration:.3f}s')
print('Release decision and download happen only after the full technical review cell.')

## Step 7: Post-Render Technical Check & Human Visual Review

This cell verifies the encoded stream and extracts a review contact sheet at the film's important transitions. It does **not** certify artistic quality automatically. Inspect the displayed frames for wave prominence, land contact, depth, gradual transformation, silhouette continuity, reflection placement, and the closing mood before treating the master as final.


In [ ]:
import os
import json
import subprocess
import shutil
import cv2
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Image as NotebookImage
from she_film.clock import DURATION, FPS, TOTAL_FRAMES

print('=' * 76)
print('POST-RENDER TECHNICAL CHECK + VISUAL REVIEW CONTACT SHEET')
print('=' * 76)
if not os.path.exists(FINAL_MASTER_MP4):
    raise FileNotFoundError(f'Final master not found: {FINAL_MASTER_MP4}')

cap = cv2.VideoCapture(FINAL_MASTER_MP4)
actual_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = float(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = actual_frames / fps if fps > 0 else 0.0
cap.release()
probe = subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type,width,height,avg_frame_rate,duration,nb_frames','-of','json',FINAL_MASTER_MP4],check=True,capture_output=True,text=True)
streams = json.loads(probe.stdout).get('streams', [])
video_stream = next((stream for stream in streams if stream.get('codec_type') == 'video'), {})
audio_stream = next((stream for stream in streams if stream.get('codec_type') == 'audio'), {})
video_fps = float(__import__('fractions').Fraction(video_stream.get('avg_frame_rate','0/1')))
audio_duration = float(audio_stream.get('duration',0.0) or 0.0)
decode = subprocess.run(['ffmpeg','-v','error','-i',FINAL_MASTER_MP4,'-f','null','-'],capture_output=True,text=True)
checks = {'frame count': actual_frames == TOTAL_FRAMES, 'duration': abs(duration - DURATION) <= 0.05, 'picture size': (width,height) == (1920,1080), 'video fps': abs(video_fps - FPS) <= 1e-6, 'H.264 video': video_stream.get('codec_name') == 'h264', 'AAC audio': audio_stream.get('codec_name') == 'aac', 'audio/video duration alignment': bool(audio_stream) and abs(audio_duration - duration) <= 0.5, 'full file decodes': decode.returncode == 0 and not decode.stderr.strip()}
stream_types = [stream.get('codec_type') for stream in streams]
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'} technical check: {label}")
print(f'  {actual_frames} frames | {fps:.3f} fps | {width}x{height} | {duration:.3f}s | streams={stream_types}')

review_points = [(24.0,'opening mushrooms'),(28.0,'mushroom 2 mature'),(31.9,'mushroom 2 hold ends'),(36.0,'mushroom 2 decay'),(40.1,'small decayed hold'),(54.0,'mushroom grounding'),(120.0,'mushroom hero hold'),(160.0,'arrival glint'),(166.0,'saucer enters'),(178.0,'saucer approach'),(182.5,'saucer landing hold'),(184.9,'landing hold end'),(198.0,'dawn departure'),(204.0,'saucer recedes'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(218.0,'forest before dinosaur entrance'),(232.0,'dinosaur entrance'),(260.0,'dinosaur forest hold'),(290.0,'pterodactyl passage'),(391.0,'caravan exit'),(397.0,'forest decay'),(400.0,'bare interval'),(406.0,'mushroom regrowth'),(414.0,'mushroom return'),(424.0,'Buddha transition'),(425.0,'7:05 dinosaur CRT + far-right noise'),(432.0,'CRT screens'),(440.0,'end credits'),(441.0,'7:21 .STORE credits')]
thumbs=[]
cap=cv2.VideoCapture(FINAL_MASTER_MP4)
for t,label in review_points:
    cap.set(cv2.CAP_PROP_POS_MSEC,t*1000); ok,bgr=cap.read()
    if not ok:
        print(f'Could not decode review frame at {t:.1f}s ({label})'); continue
    rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    thumbs.append((t,label,Image.fromarray(rgb).resize((480,270),Image.Resampling.LANCZOS)))
cap.release()
cols=3; cell_w=480; cell_h=304; rows=(len(thumbs)+cols-1)//cols
sheet=Image.new('RGB',(cols*cell_w,rows*cell_h),(16,18,23)); draw=ImageDraw.Draw(sheet)
for i,(t,label,thumb) in enumerate(thumbs):
    x=(i%cols)*cell_w; y=(i//cols)*cell_h
    draw.text((x+8,y+5),f'{t:06.1f}s  {label}',fill=(236,224,195)); sheet.paste(thumb,(x,y+28))
review_sheet_path=os.path.join(OUTPUT_DIR,'final_deliverable_visual_review.jpg')
sheet.save(review_sheet_path,quality=92)
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_RENDER_DIR,exist_ok=True); shutil.copy2(review_sheet_path,os.path.join(DRIVE_RENDER_DIR,os.path.basename(review_sheet_path)))
print(f'Visual review sheet: {review_sheet_path} ({len(thumbs)} frames)')
display(NotebookImage(filename=review_sheet_path))
print('MANUAL ART REVIEW REQUIRED: this contact sheet does not certify composition, depth, transformation quality, wave stillness, or emotional impact.')
if not all(checks.values()):
    raise RuntimeError('Technical output check failed; download is blocked. Inspect the master and rerun review.')
if DRIVE_RENDER_DIR:
    reviewed_master = os.path.join(DRIVE_RENDER_DIR, os.path.basename(FINAL_MASTER_MP4))
    shutil.copy2(FINAL_MASTER_MP4, reviewed_master)
    print('Reviewed master saved to Drive:', reviewed_master)
from google.colab import files
print('Technical checks passed. Starting browser download...')
files.download(FINAL_MASTER_MP4)
